# Advanced Python 3.10 Tutorial: Problems with Guided Solutions

## Structural Pattern Matching and `zip(..., strict=True)`

This notebook is intentionally written as a **tutorial**, not as a compact problem sheet.

Each major problem is broken into small logical steps:

1. understand the input shape;
2. identify the data contract;
3. build a small working version;
4. expose an edge case;
5. improve the implementation;
6. test the final solution;
7. discuss design choices and common mistakes.

The two main Python 3.10 topics are:

- structural pattern matching with `match` and `case`;
- strict parallel iteration with `zip(..., strict=True)`.

The examples are independent from the previous notebook and use new domains, new data models, and new exercises.

## What makes these problems advanced?

The syntax of `match` is not difficult by itself. The advanced part is deciding:

- which structure should be expressed in the pattern;
- which validation belongs in a guard;
- how case order affects correctness;
- when to capture values;
- when a wildcard is appropriate;
- when a class pattern makes a domain model clearer;
- when pattern matching is worse than a dictionary lookup or polymorphism.

Likewise, the syntax of strict `zip` is simple. The advanced part is recognizing that equal-length iteration is often a **data-integrity contract**, especially when working with generators, files, telemetry, or synchronized streams.

## Notebook conventions

- All examples use only the standard library.
- Every final solution is followed by executable assertions.
- Expected failures are caught and inspected.
- Functions validate inputs at their boundaries.
- Error messages include enough context to debug the bad input.
- The code is written for Python 3.10 or later.

In [1]:
import sys

assert sys.version_info >= (3, 10), "Python 3.10 or newer is required."
print("Running on Python", sys.version.split()[0])

Running on Python 3.13.7


# Part 1 — Structural pattern matching foundations

Before solving the larger problems, we will review the main pattern categories in a compact way.

## Literal and OR patterns

A literal pattern compares against a literal value. An OR pattern allows several alternatives to share one case body.

In [2]:
def classify_priority(priority):
    match priority:
        case "low" | "normal":
            return "non-urgent"
        case "high" | "critical":
            return "urgent"
        case _:
            return "unknown"

assert classify_priority("normal") == "non-urgent"
assert classify_priority("critical") == "urgent"
assert classify_priority("other") == "unknown"

## Capture patterns

A name inside a pattern usually captures the matched value.

That is useful, but it also creates a famous trap: an unqualified name does **not** usually compare against an existing variable with the same name.

In [3]:
def describe_number(value):
    match value:
        case 0:
            return "zero"
        case int(number):
            return f"integer {number}"
        case float(number):
            return f"float {number}"
        case _:
            return "not numeric"

assert describe_number(7) == "integer 7"
assert describe_number(2.5) == "float 2.5"

## Sequence patterns

Sequence patterns are useful when the position of each element has meaning.

They work especially well for compact command languages, protocol packets, and parsed paths.

In [4]:
def parse_simple_command(command):
    match command:
        case ["echo", str(message)]:
            return message
        case ["add", int(a), int(b)]:
            return a + b
        case ["join", *parts] if parts:
            return "-".join(map(str, parts))
        case _:
            raise ValueError(f"Unsupported command: {command!r}")

assert parse_simple_command(["echo", "hello"]) == "hello"
assert parse_simple_command(["add", 4, 9]) == 13
assert parse_simple_command(["join", "a", "b", "c"]) == "a-b-c"

## Mapping patterns

Mapping patterns express required keys without manually indexing every dictionary.

Extra keys are ignored unless they are captured with `**rest`.

In [5]:
def read_user_name(payload):
    match payload:
        case {"user": {"name": str(name), **user_extra}, **outer_extra}:
            return name, user_extra, outer_extra
        case _:
            raise ValueError("The payload does not contain a valid user name")

result = read_user_name(
    {
        "user": {"name": "Ada", "role": "admin"},
        "request_id": "R-1",
    }
)

assert result == (
    "Ada",
    {"role": "admin"},
    {"request_id": "R-1"},
)

## Class patterns

Class patterns are most valuable when the data already belongs to a domain model.

They let us match by type and destructure selected attributes.

In [6]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Temperature:
    celsius: float

def temperature_band(value):
    match value:
        case Temperature(celsius) if celsius < 0:
            return "freezing"
        case Temperature(celsius) if celsius < 25:
            return "mild"
        case Temperature():
            return "hot"
        case _:
            raise TypeError("Expected Temperature")

assert temperature_band(Temperature(-2)) == "freezing"
assert temperature_band(Temperature(18)) == "mild"
assert temperature_band(Temperature(31)) == "hot"

## Guards

A guard is an additional Boolean condition after a successful structural match.

Use guards for rules that are not naturally structural, such as numeric ranges, cross-field relationships, or membership tests.

In [7]:
def validate_interval(value):
    match value:
        case [int(start), int(end)] if start <= end:
            return start, end
        case [int(start), int(end)]:
            raise ValueError(f"Start {start} is after end {end}")
        case _:
            raise TypeError("Expected [start, end]")

assert validate_interval([3, 8]) == (3, 8)

# Problem 1 — Build a deployment configuration interpreter

A deployment tool receives configuration dictionaries from multiple clients.

The input may describe one of three actions:

- deploy an application;
- scale an existing application;
- remove an application.

We want to convert the loose dictionaries into explicit domain objects.

## Step 1 — Study the expected input shapes

A deploy request looks like:

```python
{
    "action": "deploy",
    "app": "catalog",
    "image": "catalog:2.4",
    "replicas": 3,
    "region": "eu-central"
}
```

A scale request looks like:

```python
{
    "action": "scale",
    "app": "catalog",
    "replicas": 8
}
```

A remove request looks like:

```python
{
    "action": "remove",
    "app": "catalog",
    "force": True
}
```

There may be extra keys. Those extra keys should be retained as metadata.

## Step 2 — Identify the validation rules

We will enforce these rules:

- application names must be non-empty strings;
- deployment images must contain a colon separating name and tag;
- replica counts must be integers from 1 through 100;
- region names must be non-empty strings;
- `force` must be a real Boolean, not merely a truthy value;
- unsupported actions should produce a useful error.

## Step 3 — Define domain classes

Using immutable dataclasses makes the normalized output explicit and testable.

In [8]:
from dataclasses import dataclass, field
from typing import Any

@dataclass(frozen=True)
class Deploy:
    app: str
    image: str
    replicas: int
    region: str
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class Scale:
    app: str
    replicas: int
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class Remove:
    app: str
    force: bool
    metadata: dict[str, Any] = field(default_factory=dict)

## Step 4 — Write the first matching case

We begin with the most detailed action: deployment.

The mapping pattern states the required keys. The guard handles range and content validation.

In [9]:
def parse_deployment_action_v1(payload):
    match payload:
        case {
            "action": "deploy",
            "app": str(app),
            "image": str(image),
            "replicas": int(replicas),
            "region": str(region),
            **metadata,
        } if (
            app.strip()
            and ":" in image
            and type(replicas) is int
            and 1 <= replicas <= 100
            and region.strip()
        ):
            return Deploy(
                app=app.strip(),
                image=image.strip(),
                replicas=replicas,
                region=region.strip(),
                metadata=metadata,
            )
        case _:
            raise ValueError("Not a valid deploy request")

In [10]:
sample_deploy = {
    "action": "deploy",
    "app": "catalog",
    "image": "catalog:2.4",
    "replicas": 3,
    "region": "eu-central",
    "requested_by": "release-bot",
}

deploy_result = parse_deployment_action_v1(sample_deploy)

assert deploy_result.app == "catalog"
assert deploy_result.metadata == {"requested_by": "release-bot"}
deploy_result

Deploy(app='catalog', image='catalog:2.4', replicas=3, region='eu-central', metadata={'requested_by': 'release-bot'})

## Step 5 — Add the remaining cases

The final fallback distinguishes two failure categories:

- a known action with malformed fields;
- a payload that does not even contain a usable action name.

This produces better diagnostics than a single generic error.

In [11]:
def parse_deployment_action(payload):
    match payload:
        case {
            "action": "deploy",
            "app": str(app),
            "image": str(image),
            "replicas": int(replicas),
            "region": str(region),
            **metadata,
        } if (
            app.strip()
            and ":" in image
            and type(replicas) is int
            and 1 <= replicas <= 100
            and region.strip()
        ):
            return Deploy(
                app=app.strip(),
                image=image.strip(),
                replicas=replicas,
                region=region.strip(),
                metadata=metadata,
            )

        case {
            "action": "scale",
            "app": str(app),
            "replicas": int(replicas),
            **metadata,
        } if (
            app.strip()
            and type(replicas) is int
            and 1 <= replicas <= 100
        ):
            return Scale(
                app=app.strip(),
                replicas=replicas,
                metadata=metadata,
            )

        case {
            "action": "remove",
            "app": str(app),
            "force": bool(force),
            **metadata,
        } if app.strip():
            return Remove(
                app=app.strip(),
                force=force,
                metadata=metadata,
            )

        case {"action": str(action)}:
            raise ValueError(
                f"Malformed or unsupported deployment action: {action!r}"
            )

        case _:
            raise ValueError(
                "Deployment payload must contain a string 'action' field"
            )

## Step 6 — Test successful inputs

In [12]:
actions = [
    {
        "action": "deploy",
        "app": "search",
        "image": "search:5.1",
        "replicas": 4,
        "region": "us-east",
    },
    {
        "action": "scale",
        "app": "search",
        "replicas": 10,
        "ticket": "OPS-91",
    },
    {
        "action": "remove",
        "app": "legacy-api",
        "force": False,
    },
]

parsed_actions = [parse_deployment_action(item) for item in actions]

assert parsed_actions[0] == Deploy(
    "search", "search:5.1", 4, "us-east", {}
)
assert parsed_actions[1] == Scale(
    "search", 10, {"ticket": "OPS-91"}
)
assert parsed_actions[2] == Remove(
    "legacy-api", False, {}
)

parsed_actions

[Deploy(app='search', image='search:5.1', replicas=4, region='us-east', metadata={}),
 Scale(app='search', replicas=10, metadata={'ticket': 'OPS-91'}),
 Remove(app='legacy-api', force=False, metadata={})]

## Step 7 — Test important failures

Notice the use of `type(replicas) is int`.

Without that check, `True` would match `int(replicas)` because `bool` is a subclass of `int`.

In [13]:
invalid_actions = [
    {
        "action": "deploy",
        "app": "x",
        "image": "missing-tag",
        "replicas": 2,
        "region": "eu",
    },
    {
        "action": "scale",
        "app": "x",
        "replicas": True,
    },
    {
        "action": "remove",
        "app": "",
        "force": False,
    },
]

for item in invalid_actions:
    try:
        parse_deployment_action(item)
    except ValueError as exc:
        print(type(exc).__name__ + ":", exc)
    else:
        raise AssertionError(f"Expected rejection for {item!r}")

ValueError: Malformed or unsupported deployment action: 'deploy'
ValueError: Malformed or unsupported deployment action: 'scale'
ValueError: Malformed or unsupported deployment action: 'remove'


## Problem 1 design review

This problem demonstrates:

- nested validation through mapping patterns;
- retention of unknown keys through `**metadata`;
- primitive class patterns such as `str(name)`;
- guards for ranges and content;
- specific error cases after successful structural recognition.

A long chain of dictionary indexing would work, but it would make the expected structure much harder to see.

# Problem 2 — Decode a binary-like packet protocol

A small device sends packets that have already been tokenized into Python lists.

Supported packets are:

- `["PING", request_id]`
- `["SET", key, value]`
- `["BATCH", operation_1, operation_2, ...]`
- `["DATA", sequence_number, byte_1, byte_2, ...]`

The protocol is deliberately compact, which makes sequence patterns a natural fit.

## Step 1 — Define the output types

We want parsing to return meaningful objects rather than raw lists.

In [14]:
@dataclass(frozen=True)
class PingPacket:
    request_id: str

@dataclass(frozen=True)
class SetPacket:
    key: str
    value: object

@dataclass(frozen=True)
class DataPacket:
    sequence_number: int
    payload: bytes

@dataclass(frozen=True)
class BatchPacket:
    operations: tuple[object, ...]

## Step 2 — Parse the simplest packet

The first pattern matches a two-element sequence whose first item is the literal `"PING"`.

In [15]:
def parse_ping_only(packet):
    match packet:
        case ["PING", str(request_id)] if request_id.strip():
            return PingPacket(request_id.strip())
        case _:
            raise ValueError(f"Invalid PING packet: {packet!r}")

assert parse_ping_only(["PING", "req-17"]) == PingPacket("req-17")

## Step 3 — Capture an arbitrary payload

The `SET` packet accepts any Python value in its third position.

A plain name in that position captures the value.

In [16]:
def parse_set_only(packet):
    match packet:
        case ["SET", str(key), value] if key.strip():
            return SetPacket(key.strip(), value)
        case _:
            raise ValueError(f"Invalid SET packet: {packet!r}")

assert parse_set_only(["SET", "mode", "safe"]) == SetPacket("mode", "safe")
assert parse_set_only(["SET", "limit", 25]) == SetPacket("limit", 25)

## Step 4 — Parse a starred byte payload

A starred capture gathers all remaining sequence items into a list.

The guard verifies that:

- the sequence number is a real integer from 0 through 65535;
- at least one byte exists;
- every byte is an integer from 0 through 255;
- booleans are not accepted as bytes.

In [17]:
def parse_data_only(packet):
    match packet:
        case ["DATA", int(sequence_number), *raw_bytes] if (
            type(sequence_number) is int
            and 0 <= sequence_number <= 65535
            and raw_bytes
            and all(
                type(value) is int and 0 <= value <= 255
                for value in raw_bytes
            )
        ):
            return DataPacket(sequence_number, bytes(raw_bytes))
        case _:
            raise ValueError(f"Invalid DATA packet: {packet!r}")

assert parse_data_only(["DATA", 9, 65, 66, 67]) == DataPacket(9, b"ABC")

## Step 5 — Add recursive batches

A batch contains nested packet lists. We first capture them, then recursively parse each one.

This is a useful pattern: structural matching identifies the outer shape, while ordinary Python handles recursive processing.

In [18]:
def parse_packet(packet):
    match packet:
        case ["PING", str(request_id)] if request_id.strip():
            return PingPacket(request_id.strip())

        case ["SET", str(key), value] if key.strip():
            return SetPacket(key.strip(), value)

        case ["DATA", int(sequence_number), *raw_bytes] if (
            type(sequence_number) is int
            and 0 <= sequence_number <= 65535
            and raw_bytes
            and all(
                type(value) is int and 0 <= value <= 255
                for value in raw_bytes
            )
        ):
            return DataPacket(sequence_number, bytes(raw_bytes))

        case ["BATCH", *operations] if operations:
            return BatchPacket(
                tuple(parse_packet(operation) for operation in operations)
            )

        case ["DATA", *_]:
            raise ValueError(f"Malformed DATA packet: {packet!r}")

        case ["BATCH", *_]:
            raise ValueError(f"Malformed BATCH packet: {packet!r}")

        case [str(packet_type), *_]:
            raise ValueError(f"Unsupported packet type: {packet_type!r}")

        case _:
            raise ValueError(f"Packet must be a non-empty sequence: {packet!r}")

## Step 6 — Execute nested tests

In [19]:
packet = [
    "BATCH",
    ["PING", "p-1"],
    ["SET", "mode", "fast"],
    ["DATA", 12, 1, 2, 3, 4],
    [
        "BATCH",
        ["PING", "nested"],
        ["SET", "enabled", True],
    ],
]

parsed_packet = parse_packet(packet)

assert isinstance(parsed_packet, BatchPacket)
assert parsed_packet.operations[0] == PingPacket("p-1")
assert parsed_packet.operations[2] == DataPacket(12, b"\x01\x02\x03\x04")
assert isinstance(parsed_packet.operations[3], BatchPacket)

parsed_packet

BatchPacket(operations=(PingPacket(request_id='p-1'), SetPacket(key='mode', value='fast'), DataPacket(sequence_number=12, payload=b'\x01\x02\x03\x04'), BatchPacket(operations=(PingPacket(request_id='nested'), SetPacket(key='enabled', value=True)))))

## Step 7 — Observe a failure precisely

In [20]:
bad_packets = [
    ["DATA", 1],
    ["DATA", 1, 300],
    ["DATA", True, 1, 2],
    ["BATCH"],
    ["UNKNOWN", 1, 2],
]

for bad_packet in bad_packets:
    try:
        parse_packet(bad_packet)
    except ValueError as exc:
        print(exc)
    else:
        raise AssertionError(f"Expected rejection for {bad_packet!r}")

Malformed DATA packet: ['DATA', 1]
Malformed DATA packet: ['DATA', 1, 300]
Malformed DATA packet: ['DATA', True, 1, 2]
Malformed BATCH packet: ['BATCH']
Unsupported packet type: 'UNKNOWN'


## Problem 2 design review

Sequence patterns are ideal here because packet position is part of the protocol.

A mapping pattern would be less natural because the data is not keyed. Conversely, converting every packet to a dictionary before matching would add unnecessary work and hide the real wire format.

# Problem 3 — Transform geometric shapes with class patterns

We will model circles, rectangles, and groups of shapes.

Then we will write:

- an area calculator;
- a translation function;
- a bounding-box function;
- a recursive group handler.

This problem focuses on class patterns and recursive domain models.

## Step 1 — Define the geometry model

In [21]:
from math import pi

@dataclass(frozen=True)
class Point:
    x: float
    y: float

@dataclass(frozen=True)
class Circle:
    center: Point
    radius: float

@dataclass(frozen=True)
class Rectangle:
    top_left: Point
    width: float
    height: float

@dataclass(frozen=True)
class Group:
    shapes: tuple[object, ...]

## Step 2 — Calculate area

The structure of each class selects the relevant formula.

A guard validates positive dimensions.

In [22]:
def area(shape):
    match shape:
        case Circle(Point(_, _), radius) if radius > 0:
            return pi * radius**2

        case Rectangle(Point(_, _), width, height) if width > 0 and height > 0:
            return width * height

        case Group(shapes) if shapes:
            return sum(area(item) for item in shapes)

        case Circle():
            raise ValueError("Circle radius must be positive")

        case Rectangle():
            raise ValueError("Rectangle dimensions must be positive")

        case Group():
            return 0.0

        case _:
            raise TypeError(f"Unsupported shape: {shape!r}")

In [23]:
circle = Circle(Point(2, 3), 4)
rectangle = Rectangle(Point(0, 10), 5, 2)
group = Group((circle, rectangle))

assert round(area(circle), 6) == round(pi * 16, 6)
assert area(rectangle) == 10
assert round(area(group), 6) == round(pi * 16 + 10, 6)

## Step 3 — Translate shapes

Translation changes positions but preserves dimensions.

The recursive group case applies the same operation to every nested shape.

In [24]:
def translate(shape, dx, dy):
    match shape:
        case Circle(Point(x, y), radius):
            return Circle(Point(x + dx, y + dy), radius)

        case Rectangle(Point(x, y), width, height):
            return Rectangle(Point(x + dx, y + dy), width, height)

        case Group(shapes):
            return Group(tuple(translate(item, dx, dy) for item in shapes))

        case _:
            raise TypeError(f"Unsupported shape: {shape!r}")

In [25]:
translated = translate(group, dx=10, dy=-2)

assert translated.shapes[0] == Circle(Point(12, 1), 4)
assert translated.shapes[1] == Rectangle(Point(10, 8), 5, 2)
translated

Group(shapes=(Circle(center=Point(x=12, y=1), radius=4), Rectangle(top_left=Point(x=10, y=8), width=5, height=2)))

## Step 4 — Introduce a bounding-box representation

In [26]:
@dataclass(frozen=True)
class Box:
    min_x: float
    min_y: float
    max_x: float
    max_y: float

## Step 5 — Compute bounding boxes

For a circle, the radius extends equally in all directions.

For a rectangle, we interpret `top_left.y` as the maximum y-coordinate and subtract height to obtain the minimum y-coordinate.

For a group, we recursively compute child boxes and combine their extrema.

In [27]:
def bounding_box(shape):
    match shape:
        case Circle(Point(x, y), radius) if radius > 0:
            return Box(x - radius, y - radius, x + radius, y + radius)

        case Rectangle(Point(x, y), width, height) if width > 0 and height > 0:
            return Box(x, y - height, x + width, y)

        case Group(shapes) if shapes:
            boxes = [bounding_box(item) for item in shapes]
            return Box(
                min(box.min_x for box in boxes),
                min(box.min_y for box in boxes),
                max(box.max_x for box in boxes),
                max(box.max_y for box in boxes),
            )

        case Group():
            raise ValueError("An empty group has no bounding box")

        case Circle() | Rectangle():
            raise ValueError("Shape dimensions must be positive")

        case _:
            raise TypeError(f"Unsupported shape: {shape!r}")

In [28]:
box = bounding_box(group)

assert box == Box(-2, -1, 6, 10)
box

Box(min_x=-2, min_y=-1, max_x=6, max_y=10)

## Problem 3 design review

Pattern matching is especially readable here because behavior depends on stable domain types.

However, in a large object-oriented system, methods such as `shape.area()` might be preferable. Pattern matching is strongest when:

- the operation belongs outside the classes;
- the class hierarchy is small and stable;
- several structurally different representations must be handled together.

# Problem 4 — Evaluate a recursive rule language

Suppose a filtering system stores rules as nested dictionaries.

Examples:

```python
{"eq": ["status", "active"]}
```

```python
{
    "and": [
        {"gte": ["score", 80]},
        {"in": ["region", ["eu", "us"]]}
    ]
}
```

We will evaluate these rules against ordinary records.

## Step 1 — List the supported rule forms

We support:

- equality: `{"eq": [field, expected]}`
- greater than or equal: `{"gte": [field, threshold]}`
- membership: `{"in": [field, choices]}`
- logical AND: `{"and": [rule_1, rule_2, ...]}`
- logical OR: `{"or": [rule_1, rule_2, ...]}`
- logical NOT: `{"not": rule}`

## Step 2 — Create a safe field reader

Missing fields should not crash the evaluator with a raw `KeyError`.

In [29]:
_MISSING = object()

def read_field(record, field_name):
    value = record.get(field_name, _MISSING)
    if value is _MISSING:
        raise ValueError(f"Record is missing field {field_name!r}")
    return value

## Step 3 — Implement the atomic comparisons

In [30]:
def evaluate_atomic_rule(rule, record):
    match rule:
        case {"eq": [str(field), expected]}:
            return read_field(record, field) == expected

        case {"gte": [str(field), int(threshold) | float(threshold)]}:
            actual = read_field(record, field)
            if isinstance(actual, bool) or not isinstance(actual, (int, float)):
                raise ValueError(f"Field {field!r} is not numeric")
            return actual >= threshold

        case {"in": [str(field), list(choices) | tuple(choices) | set(choices)]}:
            return read_field(record, field) in choices

        case _:
            raise ValueError(f"Unsupported atomic rule: {rule!r}")

In [31]:
record = {"status": "active", "score": 91, "region": "eu"}

assert evaluate_atomic_rule({"eq": ["status", "active"]}, record)
assert evaluate_atomic_rule({"gte": ["score", 80]}, record)
assert evaluate_atomic_rule({"in": ["region", ["eu", "us"]]}, record)

## Step 4 — Add recursive logical operators

The order of cases matters.

Atomic rules are specific, but recursive logical forms are also clearly distinguishable by their top-level keys.

In [32]:
def evaluate_rule(rule, record):
    match rule:
        case {"eq": [str(field), expected]}:
            return read_field(record, field) == expected

        case {"gte": [str(field), int(threshold) | float(threshold)]}:
            actual = read_field(record, field)
            if isinstance(actual, bool) or not isinstance(actual, (int, float)):
                raise ValueError(f"Field {field!r} is not numeric")
            return actual >= threshold

        case {"in": [str(field), list(choices) | tuple(choices) | set(choices)]}:
            return read_field(record, field) in choices

        case {"and": list(rules)} if rules:
            return all(evaluate_rule(item, record) for item in rules)

        case {"or": list(rules)} if rules:
            return any(evaluate_rule(item, record) for item in rules)

        case {"not": nested_rule}:
            return not evaluate_rule(nested_rule, record)

        case {"and": []}:
            raise ValueError("An AND rule must contain at least one child rule")

        case {"or": []}:
            raise ValueError("An OR rule must contain at least one child rule")

        case _:
            raise ValueError(f"Unsupported rule: {rule!r}")

## Step 5 — Evaluate a complex rule

In [33]:
complex_rule = {
    "and": [
        {"eq": ["status", "active"]},
        {
            "or": [
                {"gte": ["score", 90]},
                {"in": ["region", ["priority-zone"]]},
            ]
        },
        {"not": {"eq": ["blocked", True]}},
    ]
}

candidate = {
    "status": "active",
    "score": 92,
    "region": "eu",
    "blocked": False,
}

assert evaluate_rule(complex_rule, candidate) is True

## Step 6 — Filter a collection

In [34]:
candidates = [
    {"name": "Ada", "status": "active", "score": 92, "region": "eu", "blocked": False},
    {"name": "Grace", "status": "active", "score": 70, "region": "priority-zone", "blocked": False},
    {"name": "Linus", "status": "inactive", "score": 99, "region": "eu", "blocked": False},
    {"name": "Margaret", "status": "active", "score": 95, "region": "us", "blocked": True},
]

selected = [item["name"] for item in candidates if evaluate_rule(complex_rule, item)]

assert selected == ["Ada", "Grace"]
selected

['Ada', 'Grace']

## Problem 4 design review

This is a good use of recursive mapping patterns because the language itself is represented by nested mappings.

The evaluator separates:

- structural recognition;
- field access;
- type validation;
- recursive evaluation.

That separation keeps each case readable.

# Problem 5 — Normalize heterogeneous notifications

A notification service accepts multiple external formats:

1. an email mapping;
2. an SMS tuple;
3. a push-notification object;
4. a broadcast list containing nested notifications.

The goal is to normalize everything into a single `Notification` type.

## Step 1 — Define the source and target models

In [35]:
@dataclass(frozen=True)
class PushInput:
    device_id: str
    title: str
    body: str
    priority: int = 0

@dataclass(frozen=True)
class Notification:
    channel: str
    destination: str
    subject: str | None
    body: str
    priority: int

## Step 2 — Match each external format

This problem deliberately mixes mapping, sequence, and class patterns in one function.

In [36]:
def normalize_notification(item):
    match item:
        case {
            "channel": "email",
            "to": str(destination),
            "subject": str(subject),
            "body": str(body),
            "priority": int(priority),
            **email_extra,
        } if (
            "@" in destination
            and subject.strip()
            and body.strip()
            and type(priority) is int
            and 0 <= priority <= 10
        ):
            return [
                Notification(
                    "email",
                    destination.strip(),
                    subject.strip(),
                    body.strip(),
                    priority,
                )
            ]

        case (
            "sms",
            str(phone_number),
            str(body),
        ) if phone_number.strip().startswith("+") and body.strip():
            return [
                Notification(
                    "sms",
                    phone_number.strip(),
                    None,
                    body.strip(),
                    0,
                )
            ]

        case PushInput(
            device_id=str(device_id),
            title=str(title),
            body=str(body),
            priority=int(priority),
        ) if (
            device_id.strip()
            and title.strip()
            and body.strip()
            and type(priority) is int
            and 0 <= priority <= 10
        ):
            return [
                Notification(
                    "push",
                    device_id.strip(),
                    title.strip(),
                    body.strip(),
                    priority,
                )
            ]

        case ["broadcast", *items] if items:
            normalized = []
            for nested_item in items:
                normalized.extend(normalize_notification(nested_item))
            return normalized

        case _:
            raise ValueError(f"Unsupported notification: {item!r}")

## Step 3 — Test every representation

In [37]:
notifications = normalize_notification(
    [
        "broadcast",
        {
            "channel": "email",
            "to": "ada@example.com",
            "subject": "Build complete",
            "body": "Version 4.2 is ready.",
            "priority": 5,
        },
        ("sms", "+359000000000", "Deployment started"),
        PushInput(
            device_id="device-17",
            title="Alert",
            body="High memory usage",
            priority=8,
        ),
    ]
)

assert len(notifications) == 3
assert [item.channel for item in notifications] == ["email", "sms", "push"]
notifications

[Notification(channel='email', destination='ada@example.com', subject='Build complete', body='Version 4.2 is ready.', priority=5),
 Notification(channel='sms', destination='+359000000000', subject=None, body='Deployment started', priority=0),
 Notification(channel='push', destination='device-17', subject='Alert', body='High memory usage', priority=8)]

## Step 4 — Use a named mapping capture

Inside a mapping pattern, the `**rest` portion must bind to a valid name.

A clear production style is to use a descriptive name such as `**extra`, even when the current implementation does not need the additional fields. This documents that extra keys are accepted intentionally.

In [38]:
def normalize_email_only(item):
    match item:
        case {
            "channel": "email",
            "to": str(destination),
            "subject": str(subject),
            "body": str(body),
            "priority": int(priority),
            **extra,
        } if (
            "@" in destination
            and subject.strip()
            and body.strip()
            and type(priority) is int
            and 0 <= priority <= 10
        ):
            return Notification(
                "email",
                destination.strip(),
                subject.strip(),
                body.strip(),
                priority,
            )
        case _:
            raise ValueError("Invalid email notification")

assert normalize_email_only(
    {
        "channel": "email",
        "to": "a@b.com",
        "subject": "Hello",
        "body": "World",
        "priority": 1,
        "trace_id": "T-1",
    }
).channel == "email"

## Problem 5 design review

This problem shows that one `match` statement can normalize several unrelated representations.

That can be useful at an integration boundary. After normalization, the rest of the application can work with one stable type.

# Part 2 — Strict parallel iteration

The normal `zip` function stops when the shortest input ends.

That behavior is convenient when truncation is intentional. It is dangerous when all inputs are expected to describe the same records.

## A small silent-truncation example

In [39]:
devices = ["sensor-a", "sensor-b", "sensor-c"]
temperatures = [18.2, 19.1]

ordinary_result = list(zip(devices, temperatures))

assert ordinary_result == [
    ("sensor-a", 18.2),
    ("sensor-b", 19.1),
]

ordinary_result

[('sensor-a', 18.2), ('sensor-b', 19.1)]

The missing temperature for `sensor-c` did not produce an error.

In a real data pipeline, that can silently associate the wrong values with later records or discard important data.

## The strict alternative

In [40]:
try:
    list(zip(devices, temperatures, strict=True))
except ValueError as exc:
    print("Detected mismatch:", exc)
else:
    raise AssertionError("Expected a strict-zip mismatch")

Detected mismatch: zip() argument 2 is shorter than argument 1


# Problem 6 — Align telemetry streams safely

Three generators produce:

- timestamps;
- CPU percentages;
- memory percentages.

Each position is one measurement.

We must compute a list of validated measurement objects without materializing the generators merely to compare their lengths.

## Step 1 — Define the measurement type

In [41]:
from datetime import datetime, timezone

@dataclass(frozen=True)
class Measurement:
    timestamp: datetime
    cpu_percent: float
    memory_percent: float

## Step 2 — Write a range validator

Keeping validation in a helper prevents the main loop from becoming too dense.

In [42]:
def as_percentage(value, field_name):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{field_name} must be numeric")
    value = float(value)
    if not 0 <= value <= 100:
        raise ValueError(f"{field_name} must be between 0 and 100")
    return value

## Step 3 — Build the strict streaming loop

The `zip` object is lazy. Values are requested only as the loop advances.

In [43]:
def build_measurements(timestamps, cpu_values, memory_values):
    measurements = []

    try:
        aligned = zip(
            timestamps,
            cpu_values,
            memory_values,
            strict=True,
        )

        for position, (timestamp, cpu, memory) in enumerate(aligned, start=1):
            if not isinstance(timestamp, datetime):
                raise ValueError(
                    f"Timestamp at position {position} is not a datetime"
                )
            if timestamp.tzinfo is None:
                raise ValueError(
                    f"Timestamp at position {position} is timezone-naive"
                )

            measurements.append(
                Measurement(
                    timestamp=timestamp,
                    cpu_percent=as_percentage(cpu, f"CPU at position {position}"),
                    memory_percent=as_percentage(
                        memory,
                        f"memory at position {position}",
                    ),
                )
            )

    except ValueError as exc:
        if "zip()" in str(exc):
            raise ValueError(
                "Telemetry streams ended at different positions"
            ) from exc
        raise

    return measurements

## Step 4 — Test generator inputs

In [44]:
def timestamp_stream():
    yield datetime(2026, 7, 30, 10, 0, tzinfo=timezone.utc)
    yield datetime(2026, 7, 30, 10, 1, tzinfo=timezone.utc)
    yield datetime(2026, 7, 30, 10, 2, tzinfo=timezone.utc)

def cpu_stream():
    yield 25
    yield 50.5
    yield 72

def memory_stream():
    yield 40
    yield 41
    yield 43.25

measurements = build_measurements(
    timestamp_stream(),
    cpu_stream(),
    memory_stream(),
)

assert len(measurements) == 3
assert measurements[1].cpu_percent == 50.5
measurements

[Measurement(timestamp=datetime.datetime(2026, 7, 30, 10, 0, tzinfo=datetime.timezone.utc), cpu_percent=25.0, memory_percent=40.0),
 Measurement(timestamp=datetime.datetime(2026, 7, 30, 10, 1, tzinfo=datetime.timezone.utc), cpu_percent=50.5, memory_percent=41.0),
 Measurement(timestamp=datetime.datetime(2026, 7, 30, 10, 2, tzinfo=datetime.timezone.utc), cpu_percent=72.0, memory_percent=43.25)]

## Step 5 — Test misalignment

In [45]:
try:
    build_measurements(
        timestamps=timestamp_stream(),
        cpu_values=[10, 20],
        memory_values=[30, 40, 50],
    )
except ValueError as exc:
    assert "different positions" in str(exc)
    print(exc)
else:
    raise AssertionError("Expected stream-alignment failure")

Telemetry streams ended at different positions


## Problem 6 design review

Strict zipping enforces the alignment contract while preserving streaming behavior.

A failed strict zip may occur after some earlier values have already been consumed. Therefore, this pattern is safe for ordinary generators, but side-effecting streams may need checkpointing or replay support.

# Problem 7 — Create typed rows from column definitions

We receive:

- column names;
- converter functions;
- raw row values.

All three sequences must have exactly the same length.

The output should be a dictionary with converted values.

## Step 1 — See why ordinary `zip` is risky

In [46]:
names = ["id", "price", "active"]
converters = [int, float]
raw_values = ["7", "19.95", "yes"]

silently_incomplete = {
    name: converter(raw)
    for name, converter, raw in zip(names, converters, raw_values)
}

assert silently_incomplete == {"id": 7, "price": 19.95}
silently_incomplete

{'id': 7, 'price': 19.95}

The `active` column disappeared because `converters` was shorter.

That is a schema error, not a valid truncation.

## Step 2 — Define a Boolean converter

In [47]:
def parse_boolean(raw):
    match raw:
        case bool(value):
            return value
        case str(text) if text.strip().lower() in {"true", "yes", "1"}:
            return True
        case str(text) if text.strip().lower() in {"false", "no", "0"}:
            return False
        case _:
            raise ValueError(f"Cannot convert {raw!r} to bool")

## Step 3 — Build the strict row converter

In [48]:
def convert_typed_row(column_names, converters, raw_values):
    output = {}

    try:
        triples = zip(
            column_names,
            converters,
            raw_values,
            strict=True,
        )

        for position, (name, converter, raw) in enumerate(triples, start=1):
            if not isinstance(name, str) or not name.strip():
                raise ValueError(
                    f"Invalid column name at position {position}: {name!r}"
                )

            if not callable(converter):
                raise ValueError(
                    f"Converter for column {name!r} is not callable"
                )

            clean_name = name.strip()

            if clean_name in output:
                raise ValueError(f"Duplicate column name: {clean_name!r}")

            try:
                output[clean_name] = converter(raw)
            except Exception as exc:
                raise ValueError(
                    f"Column {clean_name!r} could not convert {raw!r}"
                ) from exc

    except ValueError as exc:
        if "zip()" in str(exc):
            raise ValueError(
                "Column names, converters, and values must have equal lengths"
            ) from exc
        raise

    return output

## Step 4 — Test a successful row

In [49]:
typed_row = convert_typed_row(
    column_names=["id", "price", "active"],
    converters=[int, float, parse_boolean],
    raw_values=["7", "19.95", "yes"],
)

assert typed_row == {
    "id": 7,
    "price": 19.95,
    "active": True,
}

typed_row

{'id': 7, 'price': 19.95, 'active': True}

## Step 5 — Test schema failures

In [50]:
failure_cases = [
    (
        ["id", "price", "active"],
        [int, float],
        ["7", "19.95", "yes"],
    ),
    (
        ["id", "id"],
        [int, int],
        ["1", "2"],
    ),
    (
        ["id", "active"],
        [int, parse_boolean],
        ["1", "perhaps"],
    ),
]

for names_case, converters_case, values_case in failure_cases:
    try:
        convert_typed_row(
            names_case,
            converters_case,
            values_case,
        )
    except ValueError as exc:
        print(exc)
    else:
        raise AssertionError("Expected conversion failure")

Column names, converters, and values must have equal lengths
Duplicate column name: 'id'
Column 'active' could not convert 'perhaps'


## Problem 7 design review

This implementation uses strict zip for alignment and ordinary Python for conversion.

That is a good division of responsibility:

- `zip(..., strict=True)` enforces synchronized length;
- the loop validates names and converters;
- the converter function handles field-specific parsing.

# Problem 8 — Compute a weighted grade safely

A grading system receives:

- component names;
- earned scores;
- maximum scores;
- weights.

Each position refers to the same assessment component.

We must reject:

- mismatched lengths;
- invalid scores;
- invalid weights;
- total weight not equal to 1.

## Step 1 — Define the result model

In [51]:
@dataclass(frozen=True)
class GradeComponent:
    name: str
    earned: float
    maximum: float
    weight: float
    contribution: float

@dataclass(frozen=True)
class GradeReport:
    components: tuple[GradeComponent, ...]
    final_percent: float

## Step 2 — Work through the formula

For one component:

```python
component_ratio = earned / maximum
weighted_contribution = component_ratio * weight
```

The final percentage is:

```python
sum(weighted_contribution) * 100
```

## Step 3 — Implement the strict solution

In [52]:
def weighted_grade(names, earned_scores, maximum_scores, weights):
    components = []
    total_weight = 0.0
    total_contribution = 0.0

    try:
        rows = zip(
            names,
            earned_scores,
            maximum_scores,
            weights,
            strict=True,
        )

        for position, (name, earned, maximum, weight) in enumerate(rows, start=1):
            if not isinstance(name, str) or not name.strip():
                raise ValueError(
                    f"Component name at position {position} is invalid"
                )

            numeric_values = (earned, maximum, weight)
            if any(
                isinstance(value, bool) or not isinstance(value, (int, float))
                for value in numeric_values
            ):
                raise ValueError(
                    f"Component {name!r} contains non-numeric values"
                )

            earned = float(earned)
            maximum = float(maximum)
            weight = float(weight)

            if maximum <= 0:
                raise ValueError(
                    f"Maximum score for {name!r} must be positive"
                )

            if not 0 <= earned <= maximum:
                raise ValueError(
                    f"Earned score for {name!r} must be between 0 and {maximum}"
                )

            if not 0 <= weight <= 1:
                raise ValueError(
                    f"Weight for {name!r} must be between 0 and 1"
                )

            contribution = (earned / maximum) * weight

            components.append(
                GradeComponent(
                    name=name.strip(),
                    earned=earned,
                    maximum=maximum,
                    weight=weight,
                    contribution=contribution,
                )
            )

            total_weight += weight
            total_contribution += contribution

    except ValueError as exc:
        if "zip()" in str(exc):
            raise ValueError(
                "Names, scores, maxima, and weights must have equal lengths"
            ) from exc
        raise

    if not components:
        raise ValueError("At least one grade component is required")

    if abs(total_weight - 1.0) > 1e-9:
        raise ValueError(
            f"Component weights must sum to 1.0; received {total_weight}"
        )

    return GradeReport(
        components=tuple(components),
        final_percent=total_contribution * 100,
    )

## Step 4 — Test a realistic grade

In [53]:
report = weighted_grade(
    names=["Project", "Exam", "Labs"],
    earned_scores=[88, 74, 45],
    maximum_scores=[100, 80, 50],
    weights=[0.4, 0.4, 0.2],
)

expected = (
    (88 / 100) * 0.4
    + (74 / 80) * 0.4
    + (45 / 50) * 0.2
) * 100

assert abs(report.final_percent - expected) < 1e-9
report

GradeReport(components=(GradeComponent(name='Project', earned=88.0, maximum=100.0, weight=0.4, contribution=0.35200000000000004), GradeComponent(name='Exam', earned=74.0, maximum=80.0, weight=0.4, contribution=0.37000000000000005), GradeComponent(name='Labs', earned=45.0, maximum=50.0, weight=0.2, contribution=0.18000000000000002)), final_percent=90.20000000000002)

## Step 5 — Explore integrity failures

In [54]:
try:
    weighted_grade(
        names=["Project", "Exam"],
        earned_scores=[90, 80],
        maximum_scores=[100, 100],
        weights=[0.5],
    )
except ValueError as exc:
    assert "equal lengths" in str(exc)
    print(exc)

try:
    weighted_grade(
        names=["Project", "Exam"],
        earned_scores=[90, 80],
        maximum_scores=[100, 100],
        weights=[0.4, 0.4],
    )
except ValueError as exc:
    assert "sum to 1.0" in str(exc)
    print(exc)

Names, scores, maxima, and weights must have equal lengths
Component weights must sum to 1.0; received 0.8


## Problem 8 design review

Strict zip verifies positional alignment, but it cannot verify semantic constraints such as the sum of weights.

This illustrates a broader principle:

- use strict zip for cross-sequence shape integrity;
- use explicit validation for domain integrity.

# Part 3 — Common pattern-matching mistakes

## Mistake 1 — Accidental capture instead of constant comparison

This is wrong in spirit:

```python
RED = "red"

match color:
    case RED:
        ...
```

`RED` is interpreted as a capture pattern, not as a lookup of the existing variable.

Use a literal or a qualified constant.

In [55]:
from enum import Enum

class Status(Enum):
    READY = "ready"
    FAILED = "failed"

def status_message(status):
    match status:
        case Status.READY:
            return "continue"
        case Status.FAILED:
            return "stop"
        case _:
            return "unknown"

assert status_message(Status.READY) == "continue"

## Mistake 2 — General case before specific case

The first matching case wins.

Always order cases from more specific to more general.

In [56]:
def classify_api_error(error):
    match error:
        case {"status": 404, "message": str(message), **extra}:
            return f"not found: {message}"
        case {"status": int(status), "message": str(message), **extra}:
            return f"HTTP {status}: {message}"
        case _:
            return "unknown error"

assert classify_api_error(
    {"status": 404, "message": "missing"}
) == "not found: missing"

## Mistake 3 — Treating guards as a replacement for structure

A guard should refine an already meaningful pattern.

Avoid matching everything and placing the entire parser inside one huge guard.

## Mistake 4 — Assuming strict zip checks lengths before iteration

It does not necessarily know the lengths in advance.

It detects a mismatch while advancing the iterators.

In [57]:
def traced(label, values):
    for value in values:
        print(f"{label} produced {value}")
        yield value

try:
    list(
        zip(
            traced("left", [1, 2, 3]),
            traced("right", [10, 20]),
            strict=True,
        )
    )
except ValueError as exc:
    print("Mismatch discovered after partial consumption:", exc)

left produced 1
right produced 10
left produced 2
right produced 20
left produced 3
Mismatch discovered after partial consumption: zip() argument 2 is shorter than argument 1


# Capstone — Process a synchronized import job

We will combine both Python 3.10 features.

An import job has three aligned streams:

- command payloads;
- source system names;
- ingestion timestamps.

Each command payload uses one of these shapes:

- `{"op": "create", "id": ..., "fields": {...}}`
- `{"op": "update", "id": ..., "changes": {...}}`
- `{"op": "delete", "id": ..., "reason": ...}`

The pipeline should:

1. require all three streams to have equal lengths;
2. validate timestamp and source metadata;
3. normalize each command with pattern matching;
4. produce accepted and rejected records;
5. stop immediately on stream misalignment.

## Capstone Step 1 — Define normalized commands

In [58]:
@dataclass(frozen=True)
class CreateCommand:
    record_id: int
    fields: dict[str, object]

@dataclass(frozen=True)
class UpdateCommand:
    record_id: int
    changes: dict[str, object]

@dataclass(frozen=True)
class DeleteCommand:
    record_id: int
    reason: str

## Capstone Step 2 — Define processing outcomes

In [59]:
@dataclass(frozen=True)
class AcceptedImport:
    position: int
    source: str
    timestamp: datetime
    command: CreateCommand | UpdateCommand | DeleteCommand

@dataclass(frozen=True)
class RejectedImport:
    position: int
    source: str
    timestamp: object
    payload: object
    reason: str

## Capstone Step 3 — Normalize one command

The command parser is independent from stream processing. This makes it easy to test.

In [60]:
def parse_import_command(payload):
    match payload:
        case {
            "op": "create",
            "id": int(record_id),
            "fields": dict(fields),
            **extra,
        } if (
            type(record_id) is int
            and record_id > 0
            and bool(fields)
        ):
            return CreateCommand(record_id, fields)

        case {
            "op": "update",
            "id": int(record_id),
            "changes": dict(changes),
            **extra,
        } if (
            type(record_id) is int
            and record_id > 0
            and bool(changes)
        ):
            return UpdateCommand(record_id, changes)

        case {
            "op": "delete",
            "id": int(record_id),
            "reason": str(reason),
            **extra,
        } if (
            type(record_id) is int
            and record_id > 0
            and bool(reason.strip())
        ):
            return DeleteCommand(record_id, reason.strip())

        case {"op": str(operation)}:
            raise ValueError(
                f"Malformed or unsupported import operation: {operation!r}"
            )

        case _:
            raise ValueError(
                "Import payload must contain a string 'op' field"
            )

## Capstone Step 4 — Validate metadata

We require timezone-aware timestamps and non-empty source names.

In [61]:
def validate_import_metadata(timestamp, source):
    match (timestamp, source):
        case (datetime() as valid_timestamp, str(valid_source)) if (
            valid_timestamp.tzinfo is not None
            and bool(valid_source.strip())
        ):
            return valid_timestamp, valid_source.strip()
        case _:
            raise ValueError(
                "Metadata requires a timezone-aware datetime and non-empty source"
            )

## Capstone Step 5 — Process the aligned streams

Record-level failures become rejected records.

A stream-length mismatch is different: it invalidates positional correspondence, so it becomes a fatal runtime error.

In [62]:
def process_import_job(payloads, sources, timestamps):
    results = []

    try:
        aligned = zip(
            payloads,
            sources,
            timestamps,
            strict=True,
        )

        for position, (payload, source, timestamp) in enumerate(
            aligned,
            start=1,
        ):
            try:
                valid_timestamp, valid_source = validate_import_metadata(
                    timestamp,
                    source,
                )
                command = parse_import_command(payload)
            except ValueError as exc:
                results.append(
                    RejectedImport(
                        position=position,
                        source=str(source),
                        timestamp=timestamp,
                        payload=payload,
                        reason=str(exc),
                    )
                )
            else:
                results.append(
                    AcceptedImport(
                        position=position,
                        source=valid_source,
                        timestamp=valid_timestamp,
                        command=command,
                    )
                )

    except ValueError as exc:
        if "zip()" in str(exc):
            raise RuntimeError(
                "Fatal import error: payload, source, and timestamp streams are misaligned"
            ) from exc
        raise

    return results

## Capstone Step 6 — Run a mixed import job

In [63]:
import_payloads = [
    {
        "op": "create",
        "id": 1,
        "fields": {"name": "Ada", "active": True},
    },
    {
        "op": "update",
        "id": 1,
        "changes": {},
    },
    {
        "op": "delete",
        "id": 2,
        "reason": "duplicate",
    },
]

import_sources = [
    "crm",
    "crm",
    "support",
]

import_timestamps = [
    datetime(2026, 7, 30, 12, 0, tzinfo=timezone.utc),
    datetime(2026, 7, 30, 12, 1, tzinfo=timezone.utc),
    datetime(2026, 7, 30, 12, 2, tzinfo=timezone.utc),
]

import_results = process_import_job(
    import_payloads,
    import_sources,
    import_timestamps,
)

assert isinstance(import_results[0], AcceptedImport)
assert isinstance(import_results[1], RejectedImport)
assert isinstance(import_results[2], AcceptedImport)

import_results

[AcceptedImport(position=1, source='crm', timestamp=datetime.datetime(2026, 7, 30, 12, 0, tzinfo=datetime.timezone.utc), command=CreateCommand(record_id=1, fields={'name': 'Ada', 'active': True})),
 RejectedImport(position=2, source='crm', timestamp=datetime.datetime(2026, 7, 30, 12, 1, tzinfo=datetime.timezone.utc), payload={'op': 'update', 'id': 1, 'changes': {}}, reason="Malformed or unsupported import operation: 'update'"),
 AcceptedImport(position=3, source='support', timestamp=datetime.datetime(2026, 7, 30, 12, 2, tzinfo=datetime.timezone.utc), command=DeleteCommand(record_id=2, reason='duplicate'))]

## Capstone Step 7 — Confirm fatal misalignment

In [64]:
try:
    process_import_job(
        payloads=[{"op": "delete", "id": 1, "reason": "test"}],
        sources=["admin"],
        timestamps=[],
    )
except RuntimeError as exc:
    assert "misaligned" in str(exc)
    print(exc)
else:
    raise AssertionError("Expected fatal stream misalignment")

Fatal import error: payload, source, and timestamp streams are misaligned


## Capstone design review

The capstone distinguishes two levels of failure.

### Recoverable record failure

A malformed payload or invalid metadata affects one record. We preserve it as a `RejectedImport` and continue.

### Fatal alignment failure

If one stream ends early, position no longer guarantees correspondence. Continuing would risk attaching the wrong source or timestamp to a command.

That is a system-level integrity failure, so the pipeline stops.

# Further guided exercises with solutions

These are shorter than the main problems but still include complete implementations.

## Exercise A — Parse version tuples

Accept:

- `(major, minor)`
- `(major, minor, patch)`
- `(major, minor, patch, "alpha" | "beta" | "rc")`

All numeric components must be non-negative real integers.

In [65]:
@dataclass(frozen=True)
class Version:
    major: int
    minor: int
    patch: int = 0
    stage: str = "final"

def parse_version(value):
    match value:
        case (int(major), int(minor)) if (
            type(major) is int
            and type(minor) is int
            and min(major, minor) >= 0
        ):
            return Version(major, minor)

        case (int(major), int(minor), int(patch)) if (
            all(type(item) is int for item in (major, minor, patch))
            and min(major, minor, patch) >= 0
        ):
            return Version(major, minor, patch)

        case (
            int(major),
            int(minor),
            int(patch),
            ("alpha" | "beta" | "rc") as stage,
        ) if (
            all(type(item) is int for item in (major, minor, patch))
            and min(major, minor, patch) >= 0
        ):
            return Version(major, minor, patch, stage)

        case _:
            raise ValueError(f"Invalid version: {value!r}")

assert parse_version((3, 10)) == Version(3, 10)
assert parse_version((3, 10, 2, "rc")) == Version(3, 10, 2, "rc")

## Exercise B — Strictly merge keys, values, and audit labels

In [66]:
def audited_mapping(keys, values, audit_labels):
    records = []

    try:
        for key, value, label in zip(
            keys,
            values,
            audit_labels,
            strict=True,
        ):
            records.append(
                {
                    "key": key,
                    "value": value,
                    "audit": label,
                }
            )
    except ValueError as exc:
        raise ValueError(
            "Keys, values, and audit labels must align exactly"
        ) from exc

    return records

assert audited_mapping(
    ["a", "b"],
    [1, 2],
    ["imported", "calculated"],
) == [
    {"key": "a", "value": 1, "audit": "imported"},
    {"key": "b", "value": 2, "audit": "calculated"},
]

## Exercise C — Route file-system events

In [67]:
def route_file_event(event):
    match event:
        case {"kind": "created", "path": str(path), **extra} if path:
            return ("index", path)

        case {"kind": "modified", "path": str(path), **extra} if path.endswith(".py"):
            return ("lint-and-index", path)

        case {"kind": "modified", "path": str(path), **extra} if path:
            return ("index", path)

        case {"kind": "deleted", "path": str(path), **extra} if path:
            return ("remove-index", path)

        case _:
            raise ValueError(f"Unsupported file event: {event!r}")

assert route_file_event(
    {"kind": "modified", "path": "main.py"}
) == ("lint-and-index", "main.py")

## Exercise D — Strict vector addition

In [68]:
def add_vectors(left, right):
    try:
        return [
            a + b
            for a, b in zip(left, right, strict=True)
        ]
    except ValueError as exc:
        raise ValueError("Vectors must have equal lengths") from exc

assert add_vectors([1, 2, 3], [10, 20, 30]) == [11, 22, 33]

# Final summary

Structural pattern matching is most effective when the shape of the data carries meaning.

Use it to express:

- command formats;
- nested mapping schemas;
- protocol packets;
- class-based domain models;
- recursive tree structures;
- heterogeneous integration inputs.

Use guards for:

- numeric ranges;
- relationships between captured values;
- membership tests;
- lightweight content validation.

Use `zip(..., strict=True)` whenever equal lengths are part of the contract.

It is especially valuable for:

- synchronized generators;
- table schemas;
- telemetry streams;
- aligned metadata;
- matrix or vector operations;
- multi-source data pipelines.

The most important design principle from this notebook is:

> Make structural assumptions executable.

A good pattern states the expected shape. A strict zip states the expected alignment. Both turn silent assumptions into visible, testable program behavior.

# Optional independent practice

Try extending the notebook with these tasks:

1. Add a `"pause"` packet with a positive duration to the packet protocol.
2. Add a `Triangle` class and update the geometry functions.
3. Add `"lt"` and `"contains"` operators to the rule evaluator.
4. Make telemetry processing yield measurements lazily.
5. Add a `"replace"` import command that requires both old and new field mappings.
6. Build a strict transpose function that rejects ragged matrices.
7. Add custom exception classes for record-level and alignment-level failures.

A strong solution should continue to separate:

- structural matching;
- domain validation;
- normalization;
- processing;
- error reporting.